In [1]:
# Importing Dependencies
from bertopic.representation import TextGeneration, KeyBERTInspired, MaximalMarginalRelevance
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from huggingface_hub import login
from dotenv import load_dotenv
from bertopic import BERTopic
from hdbscan import HDBSCAN
from umap import UMAP

import pandas as pd
import transformers
import numpy as np
import warnings
import torch
import yaml
import os

In [2]:
# Muting notifications from transformers
transformers.logging.set_verbosity_error()

# Filtering out potential warnings
warnings.filterwarnings(action="ignore")

# Configuring Pandas to display all columns
pd.options.display.max_columns=None

In [3]:
# Making sure GPU is available
assert torch.cuda.is_available()

# Specifying the device
device="cuda:0" if torch.cuda.is_available() else "cpu"

In [4]:
# Loading the configuration file
with open(file="../Configuration/config.yml") as yaml_file:
    config=yaml.safe_load(stream=yaml_file)

In [5]:
# Extracting the embedding model ID
emb_model_id=config.get("model_params").get("emb_model_name")

# Extracting the LLM model ID
llm_model_id=config.get("model_params").get("llm")

# Extracting the task for the LLM
llm_task=config.get("model_params").get("llm_task")

# Extracting the value for temperature
temperature=config.get("model_params").get("temperature")

# Extracting the value for top K sampling
top_k=config.get("model_params").get("top_k")

# Extracting the value nucleus sampling
top_p=config.get("model_params").get("top_p")

# Extracting the number of components
n_components=config.get("model_params").get("n_components")

# Extracting the number of clusters
cluster_size=config.get("model_params").get("cluster_size")

# Extracting the metric
metric=config.get("model_params").get("metric")

# Extracting the random state
random_state=config.get("model_params").get("random_state")

In [6]:
# Loading the dataset
df=pd.read_parquet(path="../Data/arxiv_articles.parquet.brotli", engine="fastparquet")

# Displaying the first five records
df.head()

,title,abstract,category,submission_date,authors
0,1-out-of-5 Maximin-Share Allocations Always Ex...,For four agents with nonnegative additive valu...,Economics > Theoretical Economics,Submitted on 20 Jul 2026,[[Christoph Schwerdtfeger]]
1,Equilibrium Play Without Mutual Knowledge of R...,Equilibrium play in two-player zero-sum games ...,Economics > Theoretical Economics,Submitted on 20 Jul 2026,"[[Florian Brandl, Felix Brandt]]"
2,Observable Consequences of Mental Accounting,We derive necessary and sufficient nonparametr...,Economics > Theoretical Economics,Submitted on 20 Jul 2026,"[[Laura Blow, Ian Crawford]]"
3,Belief Identification in Populations,We study the identification of belief distribu...,Economics > Theoretical Economics,Submitted on 18 Jul 2026,"[[Christopher P. Chambers, Yusufcan Masatliogl..."
4,Organization Design for Complex Worlds,I study the role of \emph{horizontal complexit...,Economics > Theoretical Economics,Submitted on 18 Jul 2026,[[Jonathan Libgober]]


In [7]:
# Creating a list of abstracts
abstracts=df.abstract.tolist()

# Printing out the number of articles
print(f"The number of articles: {len(abstracts)}")

The number of articles: 2056


In [8]:
# Loading the environment file
load_dotenv()

# Extracting the API Key
HF_TOKEN=os.getenv("HUGGINGFACE_API_KEY")

# Using the token
login(token=HF_TOKEN)

In [9]:
# Loading the model
emb_model=SentenceTransformer(model_name_or_path=emb_model_id, device=device)

# Creating the embeddings
embs=emb_model.encode(inputs=abstracts, show_progress_bar=True)

# Displaying the shape of the embeddings
embs.shape

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/65 [00:00<?, ?it/s]

(2056, 384)

In [10]:
# Applying dimensionality reduction
umap_model=UMAP(n_components=n_components, 
                metric=metric, 
                random_state=random_state)

# Fitting the abstracts
embs_reduced=umap_model.fit_transform(X=embs)

# Displaying the shape
embs_reduced.shape

(2056, 25)

In [11]:
# Building a clustering model
hdbscan_model=HDBSCAN(min_cluster_size=cluster_size)

# Fitting the embeddings
hdbscan_model.fit(X=embs_reduced)

,min_cluster_size,25
,min_samples,None
,cluster_selection_epsilon,0.0
,cluster_selection_persistence,0.0
,max_cluster_size,0
,metric,'euclidean'
,alpha,1.0
,p,None
,algorithm,'best'
,leaf_size,40
,memory,Memory(location=None)


In [12]:
# Creating an array of clusters
clusters=hdbscan_model.labels_

# Printing the number of clusters
print(f"The number of clusters: {len(set(clusters))}")

The number of clusters: 8


In [13]:
# Looping through the first cluster
for i in np.where(clusters==0)[0][:3]:
    # Printing out the abstract
    print(df.loc[i].get("title"))

Geometric Approach to Quantum Theory. L-functionals
Geometric quantum discord in the black hole quantum atmosphere
Parker Solar Probe Observations of Preferential Heating of Protons over Alpha Particles near Turbulent Coherent Structures


In [14]:
prompt="""I have a topic that contains the following documents:
[DOCUMENTS]
The topic is described by the following keywords: '[KEYWORDS]'.
Based on the documents and keywords, create a highly concise label for this topic.
Output ONLY the topic label and nothing else.
Topic Label:"""

# Loading the model
generator=pipeline(task=llm_task,
                   model=llm_model_id,
                   device_map=device,
                   max_new_tokens=50,
                   return_full_text=False,
                   do_sample=True,
                   temperature=temperature,
                   top_k=top_k,
                   top_p=top_p)

# Calculating the average document length
avg_doc_length=round(number=sum([len(i.split()) for i in abstracts])/len(abstracts))

# Instantiating the LLM model
llm=TextGeneration(model=generator,
                   prompt=prompt,
                   doc_length=avg_doc_length,
                   tokenizer="whitespace")

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

In [15]:
# Creating a list of representation models
representation_chain=[KeyBERTInspired(), MaximalMarginalRelevance(diversity=0.3)]

# Creating a dictionary
representation_dict={"Main": representation_chain, "LLM": llm}

# Building the topic model
topic_model=BERTopic(embedding_model=emb_model,
                     umap_model=umap_model,
                     hdbscan_model=hdbscan_model,
                     representation_model=representation_dict,
                     verbose=True)

# Fitting the abstracts and embeddings
topic_model.fit(documents=abstracts, embeddings=embs)

2026-07-22 22:45:19,185 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-07-22 22:45:24,571 - BERTopic - Dimensionality - Completed ✓
2026-07-22 22:45:24,574 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-07-22 22:45:24,625 - BERTopic - Cluster - Completed ✓
2026-07-22 22:45:24,640 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 8/8 [04:09<00:00, 31.19s/it]
2026-07-22 22:49:35,779 - BERTopic - Representation - Completed ✓


In [16]:
# Displaying the topics
topic_model.get_topic_info()

,Topic,Count,Name,Representation,LLM,Representative_Docs
0,-1,378,-1_algorithm_problem_scale_complexity,"[algorithm, problem, scale, complexity, method...",[Optimization Algorithms and Plasma Modeling i...,[Radio-frequency (RF) capacitively coupled pla...
1,0,842,0_ai_model_models_language,"[ai, model, models, language, visual, reasonin...",[Autonomous Driving and Large Language Model I...,"[World models, especially based on JEPA archit..."
2,1,271,1_quantum_dynamics_topological_phonon,"[quantum, dynamics, topological, phonon, symme...",[Quantum Entanglement and Magnetic Topological...,"[In the two-dimensional limit, the intrinsic m..."
3,2,182,2_gravitational_galaxies_cosmological_gravity,"[gravitational, galaxies, cosmological, gravit...",[Compact Star Physics and Modified Gravity The...,[In this work we reconstruct two subclasses of...
4,3,172,3_conjecture_mathbb_graphs_finite,"[conjecture, mathbb, graphs, finite, mathfrak,...",[Graph Theory and Group Actions in Discrete Ma...,[We consider corner-cutting schemes for closed...
5,4,97,4_dynamics_stochastic_nonlinear_approximation,"[dynamics, stochastic, nonlinear, approximatio...",[Stochastic Dynamics in Chemical Reaction Syst...,[We investigate the stochastic stability of ma...
6,5,61,5_regularity_let_bounded_convex,"[regularity, let, bounded, convex, curvature, ...",[Convex Domain Analysis in Eigenvalue Problems...,[We investigate the relation between minimizer...
7,6,53,6_beamforming_sensing_antenna_wireless,"[beamforming, sensing, antenna, wireless, opti...",[mmWave MIMO Channel Estimation and ISAC Syste...,[Integrated sensing and communication (ISAC) i...


In [17]:
# Extracting the top N words for the first topic
topic_model.get_topic(topic=0)

[('ai', np.float64(0.8833746842548477)),
 ('model', np.float64(0.8666162287381468)),
 ('models', np.float64(0.8641878871495013)),
 ('language', np.float64(0.8640681299029753)),
 ('visual', np.float64(0.8629583238206235)),
 ('reasoning', np.float64(0.8594898919064936)),
 ('learning', np.float64(0.8590134995489143)),
 ('task', np.float64(0.8570880929517076)),
 ('tasks', np.float64(0.8514897751042307)),
 ('memory', np.float64(0.8495376147140076))]

In [18]:
# Searching for topics based on a keyword
topic_model.find_topics(search_term="Machine Learning")

([0, -1, 6, 1, 4],
 [np.float64(0.8933300473639436),
  np.float64(0.8872034450773676),
  np.float64(0.8514295990385345),
  np.float64(0.8408402417763183),
  np.float64(0.8407872047173448)])

In [19]:
# Visualizing the topics
topic_model.visualize_documents(docs=df.title.tolist(),
                                reduced_embeddings=embs_reduced,
                                width=1200,
                                hide_annotations=True)

In [20]:
# Visualizing the keywords
topic_model.visualize_barchart()

In [21]:
# Visualizing the topics
topic_model.visualize_heatmap()

In [22]:
# Visualize the potential hierarchical structure of topics
topic_model.visualize_hierarchy()

In [23]:
# Extracting the top key words for each topic
topic_model.topic_representations_

{-1: [('algorithm', np.float64(0.8600640764227004)),
  ('problem', np.float64(0.8595577642985737)),
  ('scale', np.float64(0.8585202505396747)),
  ('complexity', np.float64(0.8549199093860355)),
  ('methods', np.float64(0.853718044231463)),
  ('finite', np.float64(0.8517217489006936)),
  ('model', np.float64(0.8517189669244334)),
  ('level', np.float64(0.8510152379102564)),
  ('models', np.float64(0.8503090294216746)),
  ('graph', np.float64(0.8502302338131149))],
 0: [('ai', np.float64(0.8833746842548477)),
  ('model', np.float64(0.8666162287381468)),
  ('models', np.float64(0.8641878871495013)),
  ('language', np.float64(0.8640681299029753)),
  ('visual', np.float64(0.8629583238206235)),
  ('reasoning', np.float64(0.8594898919064936)),
  ('learning', np.float64(0.8590134995489143)),
  ('task', np.float64(0.8570880929517076)),
  ('tasks', np.float64(0.8514897751042307)),
  ('memory', np.float64(0.8495376147140076))],
 1: [('quantum', np.float64(0.8769355888931059)),
  ('dynamics', np.